# Etna Cause-Trigger Analysis

This notebook applies the Cause-Trigger workflow to the Etna 2008 case study. The full dataset is loaded as the constructed data product, but the algorithm is run only on a fixed case-study interval around the Wenchuan teleseismic arrival.

The split into \(I_1\) and \(I_2\) is selected automatically from the effect variable. The known event time is shown only for physical interpretation.

## 1. Imports, paths, and case-study interval

In [ ]:
from pathlib import Path
import sys
import warnings
import pandas as pd

warnings.filterwarnings("ignore", message="No frequency information was provided")
warnings.filterwarnings("ignore", message="A date index has been provided")

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 40)

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "etna"

for path in [SRC_DIR, NOTEBOOK_DIR]:
    if str(path) not in sys.path:
        sys.path.append(str(path))

from cause_trigger_etna import (
    COMPACT_RUN_SPECS,
    DEFAULT_RUN_SPECS,
    EFFECT,
    EtnaWorkflowConfig,
    case_study_interval,
    compact_grid_errors,
    compact_grid_outputs,
    expand_pair_outputs,
    load_model_frame,
    model_overview,
    plot_effect_with_split,
    reference_parameter_table,
    run_sensitivity_grid,
    split_diagnostics,
    compact_comparison,
    compact_diagnostics,
    prepare_case_model_frame,
    run_suite,
)

from cause_trigger import summarize_pair_recurrence

ETNA_PATH = DATA_DIR / "etna_dataset.csv"
EVENT_TIME = pd.Timestamp("2008-05-12 06:28:00", tz="UTC")

In [ ]:
CASE_PRE_DAYS = 3
CASE_POST_HOURS = 72

# These constraints allow an actual split search.
MIN_I1_LENGTH = 48
MIN_I2_LENGTH = 48
MAX_LAGS = 12

run_specs = DEFAULT_RUN_SPECS # "DEFAULT_RUN_SPECS" or "COMPACT_RUN_SPECS". compact considers tau=0 contemporaneous trigger candidates
cond_ind_test = "robust_parcorr" # "parcorr" or "robust_parcorr"
pcmci_pc_alpha = 0.05
# PC-stage conditional-independence threshold.
# For PCMCI+, this is the principal graph-selection threshold.

pcmci_alpha_level = 0.05
# Final q/p threshold used for lagged PCMCI selection and for storing
# significant PCMCI+ contemporaneous diagnostics.

pcmci_fdr_method = "fdr_bh"
# Multiple-testing correction applied by the selected backend.

ETNA_MODEL_COLUMNS = [
    "teleseismic",
    "local_event_rate_state",
    "local_event_rate_response",
    "CO2_3",
    "rainfall_mm",
    "pressure_drop",
]

X_full = load_model_frame(
    ETNA_ANALYSIS_PATH,
    include_columns=ETNA_MODEL_COLUMNS,
    require_complete=False,
)

X_case_raw = case_study_interval(
    X_full,
    EVENT_TIME,
    pre_days=CASE_PRE_DAYS,
    post_hours=CASE_POST_HOURS,
)

X = prepare_case_model_frame(X_case_raw)

display(pd.DataFrame([model_overview(X, EFFECT)]))

## 2. Reference parameters and split audit

This section reports two checks before running the causal backends.

`reference_parameters` shows the VAR-AIC and VAR-BIC lag suggestions. These are metadata only; they do not select the final interpretation.

`split_diagnostics` shows the automatic Cause-Trigger split: `split_time`, lengths of \(I_1\) and \(I_2\), the target mean change, distance to the known event time, and whether the split is a boundary artefact.

In [ ]:
reference_parameters = reference_parameter_table(
    X,
    EFFECT,
    max_lags=MAX_LAGS,
    fallback_lag=1,
    fallback_distribution="gaussian",
)
display(reference_parameters)

aic_row = reference_parameters.loc[
    reference_parameters["lag_method"].eq("VAR-AIC")
].iloc[0]

selected_lag = int(aic_row["selected_lag"])
selected_distribution = "gaussian"
# Fixed for transformed, standardized data. Gamma and inverse Gaussian
# require positive support.

workflow = EtnaWorkflowConfig(
    effect=EFFECT,
    event_time=EVENT_TIME,
    selected_lag=selected_lag,
    max_lags=MAX_LAGS,
    min_I1_length=MIN_I1_LENGTH,
    min_I2_length=MIN_I2_LENGTH,
    distribution=selected_distribution,
    parameter_source="paper_VAR_AIC",
    pcmci_pc_alpha=pcmci_pc_alpha,
    pcmci_alpha_level=pcmci_alpha_level,
    pcmci_fdr_method=pcmci_fdr_method,
    pcmci_cond_ind_test=cond_ind_test,
    pcmci_plus_use_contemporaneous_triggers=False,
    run_specs=run_specs,
)

split_row = split_diagnostics(
    X,
    EFFECT,
    event_time=EVENT_TIME,
    min_I1_length=workflow.min_I1_length,
    min_I2_length=workflow.min_I2_length,
)

display(pd.DataFrame([split_row]))

plot_effect_with_split(
    X,
    EFFECT,
    split_row,
    event_time=EVENT_TIME,
)

# Primary analysis:
# HMML, VAR-AIC lag, paper split and paper moderation test.
primary_specs = (
    {
        "run": "hmml_paper_baseline",
        "backend": "hmml",
    },
)

primary_results, primary_comparison, primary_diagnostics = run_suite(
    X,
    workflow,
    run_specs=primary_specs,
    lag=selected_lag,
    distribution=selected_distribution,
)

display(compact_comparison(primary_comparison))
display(compact_diagnostics(primary_diagnostics))

## 3. Secondary backend and lag robustness grid

The HMML run at the VAR-AIC lag above is the primary paper-compatible
analysis. The following grid is a secondary backend-comparison
analysis. Recurrence across lags is descriptive and does not replace the VAR-AIC parameter selection.

In [ ]:
lag_grid = run_sensitivity_grid(
    X,
    workflow,
    run_specs=run_specs,
    lags=range(1, workflow.max_lags + 1),
    distributions=(workflow.distribution,),
    cond_ind_test=cond_ind_test,
)

pair_rows = expand_pair_outputs(lag_grid)
signal_rows = compact_grid_outputs(lag_grid, mode="signals")
error_rows = compact_grid_errors(lag_grid)
pair_recurrence = summarize_pair_recurrence(pair_rows)

print(f"All grid rows: {len(lag_grid)}")
print(f"Rows with accepted pairs: {len(pair_rows)}")
print(f"Rows with any backend signal: {len(signal_rows)}")
print(f"Rows with backend errors: {len(error_rows)}")

display(pair_rows)
display(pair_recurrence)
display(signal_rows)

if not error_rows.empty:
    display(error_rows)